# Comparativa Visual: PCA 5 vs PCA 12

Este notebook compara la estructura geométrica de los clústeres cuando proyectamos los datos a 5 dimensiones (perdiendo el 46% de varianza biológica) frente a 12 dimensiones (reteniendo el 80% de varianza biológica).

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

## 1. Carga de Datos y Preprocesamiento

In [2]:
DATA_PATH = r"c:\Universidad\mineria\clustering\data\20k\clustering_feature_view.csv"
MODEL_DIR = r"c:\Universidad\machine_learning_service\models"

df = pd.read_csv(DATA_PATH)
feature_columns = joblib.load(os.path.join(MODEL_DIR, "feature_columns.pkl"))
preprocessor = joblib.load(os.path.join(MODEL_DIR, "preprocessor.pkl"))

df_filtered = df[feature_columns].copy()
X_scaled = preprocessor.transform(df_filtered)

## 2. Modelo 1: PCA a 5 Componentes (Modelo Actual en Producción)

In [3]:
pca_5 = PCA(n_components=5, random_state=42)
X_pca_5 = pca_5.fit_transform(X_scaled)
var_5 = sum(pca_5.explained_variance_ratio_)

kmeans_5 = KMeans(n_clusters=4, init='k-means++', n_init=10, max_iter=300, random_state=42)
labels_5 = kmeans_5.fit_predict(X_pca_5)
sil_5 = silhouette_score(X_pca_5, labels_5)

print(f"Varianza retenida: {var_5:.2%}")
print(f"Silhouette Score: {sil_5:.4f}")

Varianza retenida: 53.08%
Silhouette Score: 0.2578


### 2.1 Visualización 3D (5 Componentes)
Graficamos los 3 primeros componentes (donde hay más varianza) para ver cómo K-Means separó los grupos.

In [4]:
df_plot_5 = pd.DataFrame(X_pca_5[:, :3], columns=['PC1', 'PC2', 'PC3'])
df_plot_5['Cluster'] = labels_5.astype(str)

# Usar una muestra de 2000 puntos para que el renderizado 3D sea fluido
sample_5 = df_plot_5.sample(2000, random_state=42)

fig_5 = px.scatter_3d(sample_5, x='PC1', y='PC2', z='PC3', color='Cluster', 
                      title=f"Clusters en 5 Componentes (Varianza: {var_5:.2%})",
                      color_discrete_sequence=px.colors.qualitative.Plotly,
                      opacity=0.7)
fig_5.update_traces(marker=dict(size=4))
fig_5.show()

## 3. Modelo 2: PCA a 12 Componentes (Recomendación Clínica)

In [5]:
pca_12 = PCA(n_components=12, random_state=42)
X_pca_12 = pca_12.fit_transform(X_scaled)
var_12 = sum(pca_12.explained_variance_ratio_)

kmeans_12 = KMeans(n_clusters=4, init='k-means++', n_init=10, max_iter=300, random_state=42)
labels_12 = kmeans_12.fit_predict(X_pca_12)
sil_12 = silhouette_score(X_pca_12, labels_12)

print(f"Varianza retenida: {var_12:.2%}")
print(f"Silhouette Score: {sil_12:.4f}")

Varianza retenida: 80.18%
Silhouette Score: 0.1609


### 3.1 Visualización 3D (12 Componentes)
Al tener 12 ejes, graficamos los 3 primeros componentes. Visualmente puede parecer más caótico porque hay mucha información biomédica oculta en los 9 ejes invisibles que Plotly no puede dibujar.

In [6]:
df_plot_12 = pd.DataFrame(X_pca_12[:, :3], columns=['PC1', 'PC2', 'PC3'])
df_plot_12['Cluster'] = labels_12.astype(str)

sample_12 = df_plot_12.sample(2000, random_state=42)

fig_12 = px.scatter_3d(sample_12, x='PC1', y='PC2', z='PC3', color='Cluster', 
                      title=f"Clusters en 12 Componentes (Varianza: {var_12:.2%})",
                      color_discrete_sequence=px.colors.qualitative.Plotly,
                      opacity=0.7)
fig_12.update_traces(marker=dict(size=4))
fig_12.show()